# Task 1 : Triage Agent 

In [21]:
import pandas as pd 
import numpy as np

print("Lets  start")

Lets  start


In [22]:
knowledge_base_clean = pd.read_csv("../data/knowledge_base_cleaned.csv")

print("Shape:",knowledge_base_clean.shape)
print("\nSeverity value counts :\n")
print(knowledge_base_clean['severity'].value_counts())


Shape: (29161, 9)

Severity value counts :

severity
normal      15110
Major        5775
Minor        2171
major        1689
critical      964
minor         919
Critical      575
trivial       456
blocker       408
Trivial       403
Unknown       353
Blocker       338
Name: count, dtype: int64


In [24]:
knowledge_base_clean.head()

,bug_id,title,description,severity,resolution,status,source_dataset,stack_trace,product_component
0,BUGZILLA-294734,Emergency 2.16.10 Release,2.16.9 is broken -- many users can't enter bug...,blocker,fixed,resolved,Mozilla,NaN,BUGZILLA
1,OTHER_APPLICATIONS-363323,DOM View is really inefficient with setting wh...,From comment in url: Current code: menuitem ->...,normal,fixed,resolved,Mozilla,NaN,OTHER_APPLICATIONS
2,SUPPORT.MOZILLA.ORG-398246,Add support for custom cookies and cache headers,Adding support for custom headers and cookie n...,blocker,fixed,resolved,Mozilla,NaN,SUPPORT.MOZILLA.ORG
3,OTHER_APPLICATIONS-318859,DCC functionality in ChatZilla isn't functional.,User-Agent: Mozilla/5.0 (Macintosh U PPC Mac O...,normal,fixed,resolved,Mozilla,NaN,OTHER_APPLICATIONS
4,DEVELOPER.MOZILLA.ORG-416840,Fix and cruft,Since we removed the breadcrumbs and title-ove...,normal,fixed,resolved,Mozilla,NaN,DEVELOPER.MOZILLA.ORG


In [23]:
knowledge_base_clean['severity'] = knowledge_base_clean['severity'].str.lower()
print(knowledge_base_clean['severity'].value_counts())


severity
normal      15110
major        7464
minor        3090
critical     1539
trivial       859
blocker       746
unknown       353
Name: count, dtype: int64


In [7]:
severity_mapping = {
    'blocker':'Critical',
'critical':'Critical',
'major':'High',
'normal':'Medium',
'minor':'Low',
'trivial':'Low',
'unknown':'Medium',

}

knowledge_base_clean['severity_mapped'] = knowledge_base_clean['severity'].map(severity_mapping)
print(knowledge_base_clean['severity_mapped'].value_counts())

severity_mapped
Medium      15463
High         7464
Low          3949
Critical     2285
Name: count, dtype: int64


In [8]:
knowledge_base_clean.to_csv("../data/knowledge_base_with_severity.csv", index = False)
print("Saved")

Saved


## New plan 

In [ ]:
import sys
!{sys.executable}  -m pip install groq python-dotenv

In [ ]:
import os

print("Current notebook working directory:", os.getcwd())
print("\nLooking for .env in parent folder...")
print("Exists:", os.path.exists("../.env"))

In [17]:
env_content = "API_KEY_NAME= API_KEY"
with open  ("../.env",'w') as f:
    f.write(env_content)
print("Created .env file ")

Created .env file 


In [18]:
import os
print("Exists:", os.path.exists("../.env"))

Exists: True


In [15]:
from dotenv import load_dotenv
import os 

load_dotenv("../.env",override=True)

api_key = os.getenv("bug-analyzer-triage")

if api_key:
    print("API loaded successfully . Starrts with :",api_key[:8] , "...")
else:
    print("API key not found - check your .env file location/content")

API loaded successfully . Starrts with : gsk_mpz3 ...


In [32]:
knowledge_base_clean = pd.read_csv("../data/knowledge_base_with_severity.csv")

print(knowledge_base_clean.columns.tolist())

['bug_id', 'title', 'description', 'severity', 'resolution', 'status', 'source_dataset', 'stack_trace', 'product_component', 'severity_mapped']


In [42]:
from groq import Groq
import json

client  =  Groq (api_key=api_key)

# system_prompt ="""
# You are a bug triage assistant. You are given the title, description, and stack trace of a submitted bug, and you classify it by severity, priority, and affected component, with confidence scoring and reasoning.

# Reply ONLY in valid JSON, with this exact structure:
# {"bug_id": "...", "severity": "Critical", "confidence": 0.85, "reasoning": "...", "priority": "...", "component": "Login"}

# Field definitions:
# - bug_id: if provided, use it; otherwise use "unknown"
# - severity: must be exactly one of Critical, High, Medium, Low — no other words
# - confidence: a score from 0 to 1 representing how confident you are in this classification
# - reasoning: a brief explanation of why you classified it this way
# - priority: how urgently this needs to be addressed — Immediate, High, Medium, or Low (related to but different from severity: severity is impact, priority is urgency)
# - component: which part/module of the software seems affected, inferred from the content (e.g. "Login", "Database", "UI/Rendering") — do not restrict to a fixed list

# If you encounter other severity-like words, map them using this table:
# blocker/critical -> Critical, major -> High, normal/unknown -> Medium, minor/trivial -> Low


# """


system_prompt = """You are a bug triage assistant. You are given the title, description, and stack trace of a submitted bug, and you classify it by severity, priority, and affected component, with confidence scoring and reasoning.

IMPORTANT CALIBRATION: Most everyday bugs are Medium severity, not Critical or High. Reserve Critical/High for bugs that cause crashes, data loss, security issues, or completely block core functionality. Reserve Low for cosmetic issues, minor inconveniences, or feature requests. The MAJORITY of real-world bugs are routine Medium severity — do not over-classify ordinary bugs as urgent just because they mention words like "error" or "exception."

Examples for calibration:
- "App crashes on startup, no workaround" -> Critical
- "Login fails for all users" -> Critical  
- "Button text is misaligned by 2px" -> Low
- "Feature request: add dark mode" -> Low
- "Export function occasionally produces incorrect date format" -> Medium
- "Search results are slow to load under heavy traffic" -> Medium

Reply ONLY in valid JSON, with this exact structure:
{"bug_id": "...", "severity": "Critical", "confidence": 0.85, "reasoning": "...", "priority": "...", "component": "Login"}

Field definitions:
- bug_id: if provided, use it; otherwise use "unknown"
- severity: must be exactly one of Critical, High, Medium, Low — no other words
- confidence: a score from 0 to 1 representing how confident you are in this classification
- reasoning: a brief explanation of why you classified it this way
- priority: how urgently this needs to be addressed — Immediate, High, Medium, or Low (related to but different from severity: severity is impact, priority is urgency)
- component: which part/module of the software seems affected, inferred from the content (e.g. "Login", "Database", "UI/Rendering") — do not restrict to a fixed list

If you encounter other severity-like words, map them using this table:
blocker/critical -> Critical, major -> High, normal/unknown -> Medium, minor/trivial -> Low
"""


test_bug= knowledge_base_clean.iloc[0]
user_message = f"Title:{test_bug['title']} \nDescription:{test_bug['description']} "

response = client.chat.completions.create(
    model = 'llama-3.1-8b-instant',
    messages=[
        {"role":"system","content":system_prompt},
        {"role":"user","content":user_message}
    ],
    temperature=0.2
)

result = response.choices[0].message.content
print(result)


{"bug_id": "unknown", "severity": "Critical", "confidence": 0.95, "reasoning": "The issue prevents many users from entering bugs, which is a core functionality. This is a clear blocker for the current release.", "priority": "Immediate", "component": "Release Management"}


In [30]:
knowledge_base_clean.columns

Index(['bug_id', 'title', 'description', 'severity', 'resolution', 'status',
       'source_dataset', 'stack_trace', 'product_component'],
      dtype='str')

In [31]:
print("Actual bug title:", test_bug['title'])
print("Actual bug description:", test_bug['description'][:200])
print("\nActual historical severity (ground truth):", test_bug['severity'])
print("\nLLM's predicted severity:", "Critical")

Actual bug title: Emergency 2.16.10 Release
Actual bug description: 2.16.9 is broken -- many users can't enter bugs on it particularly not from a fresh install. So we need to pull 2.16.9 and post a 2.16.10 instead.

Actual historical severity (ground truth): blocker

LLM's predicted severity: Critical


In [37]:
print("Actual bug title:", test_bug['title'])
print("Actual bug description:", test_bug['description'][:200])
print("\nActual historical severity (ground truth):", test_bug['severity_mapped'])
print("\nLLM's predicted severity:", "Critical")

Actual bug title: Emergency 2.16.10 Release
Actual bug description: 2.16.9 is broken -- many users can't enter bugs on it particularly not from a fresh install. So we need to pull 2.16.9 and post a 2.16.10 instead.

Actual historical severity (ground truth): Critical

LLM's predicted severity: Critical


In [39]:
import time

severities = ['Critical', 'High', 'Medium', 'Low']
sample_bugs = pd.concat([
    knowledge_base_clean[knowledge_base_clean['severity_mapped'] == s].sample(1, random_state=42)
    for s in severities
]).reset_index(drop=True)

print("Testing on", len(sample_bugs), "sample bugs (one per severity level)\n")
print(sample_bugs[['title', 'severity_mapped']])
for idx, bug in sample_bugs.iterrows():
    user_message = f"Title: {bug['title']}\nDescription: {bug['description']}\nStack Trace: Not available"

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0.2
    )

    result = response.choices[0].message.content

    print(f"--- Bug {idx+1} ---")
    print("Title:", bug['title'])
    print("Actual severity:", bug['severity_mapped'])
    print("LLM response:", result)
    print()

    time.sleep(1)  # small pause to avoid hitting rate limits

Testing on 4 sample bugs (one per severity level)

                                               title severity_mapped
0  support project inheritence with flat project ...        Critical
1  Client use Axis-cpp, string array type not com...            High
2      [DND] Gtk ImageTransfer prefers JPEG over PNG          Medium
3                              IoSession.isClosing()             Low
--- Bug 1 ---
Title: support project inheritence with flat project layout
Actual severity: Critical
LLM response: {
  "bug_id": "unknown",
  "severity": "High",
  "confidence": 0.9,
  "reasoning": "The bug description mentions a specific limitation in project inheritance due to a flat directory structure, which suggests a significant impact on project functionality.",
  "priority": "High",
  "component": "Project Management"
}

--- Bug 2 ---
Title: Client use Axis-cpp, string array type not compatible with Axic at server
Actual severity: High
LLM response: {
  "bug_id": "unknown",
  "severity": "C

In [47]:
import random

random.seed(42)
test_sample = knowledge_base_clean.sample(30, random_state=42).reset_index(drop=True)

severity_order = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}

exact_matches = 0
adjacent_matches = 0
results_log = []

for idx, bug in test_sample.iterrows():
    user_message = f"Title: {bug['title']}\nDescription: {bug['description']}\nStack Trace: Not available"

    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=0.2
        )
        result = json.loads(response.choices[0].message.content)
        predicted = result.get('severity', 'Medium')
    except Exception as e:
        predicted = 'Medium'
        print(f"Error on bug {idx}: {e}")

    actual = bug['severity_mapped']

    is_exact = (predicted == actual)
    is_adjacent = abs(severity_order.get(predicted, 1) - severity_order.get(actual, 1)) <= 1

    if is_exact:
        exact_matches += 1
    if is_adjacent:
        adjacent_matches += 1

    results_log.append({'title': bug['title'], 'actual': actual, 'predicted': predicted, 'exact': is_exact})

    time.sleep(1)

print(f"\nExact match accuracy: {exact_matches}/{len(test_sample)} = {exact_matches/len(test_sample)*100:.1f}%")
print(f"Adjacent (±1 level) accuracy: {adjacent_matches}/{len(test_sample)} = {adjacent_matches/len(test_sample)*100:.1f}%")

Error on bug 24: Expecting value: line 1 column 1 (char 0)
Error on bug 26: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kxfy6ngtfr49wg8hhv46tepc` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5245, Requested 768. Please try again in 130ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Exact match accuracy: 18/30 = 60.0%
Adjacent (±1 level) accuracy: 25/30 = 83.3%


In [48]:
results_df = pd.DataFrame(results_log)
print("Prediction distribution:")
print(results_df['predicted'].value_counts())
print("\nActual distribution (in this sample):")
print(results_df['actual'].value_counts())

print("\n--- Biggest misses (predicted far from actual) ---")
severity_order = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
results_df['distance'] = results_df.apply(
    lambda r: abs(severity_order.get(r['predicted'], 1) - severity_order.get(r['actual'], 1)), axis=1
)
print(results_df[results_df['distance'] >= 2][['title', 'actual', 'predicted']])

Prediction distribution:
predicted
Medium      22
Low          4
Critical     4
Name: count, dtype: int64

Actual distribution (in this sample):
actual
Medium      19
Low          5
Critical     3
High         3
Name: count, dtype: int64

--- Biggest misses (predicted far from actual) ---
                                                title    actual predicted
0          Add Mockito and AssertJ in target platform  Critical       Low
7   SWTException: Invalid thread access on disconn...       Low  Critical
10  Access Error trying to access vpg when creatin...    Medium  Critical
19  std::numeric_limits ::traps = false when integ...       Low  Critical
21             missing metadata errors on the console  Critical    Medium


In [16]:
from groq import Groq
import json
client = Groq(api_key=api_key)
import time

def call_llm_with_retry(system_prompt, user_message, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_message}
                ],
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait_time = (attempt + 1) * 3  # wait 3s, then 6s, then 9s
                print(f"Rate limited, waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"Non-rate-limit error: {e}")
                break
    return None  # all retries failed

In [50]:
def triage_agent(title, description, stack_trace="Not available", bug_id="unknown"):
    user_message = f"Title: {title}\nDescription: {description}\nStack Trace: {stack_trace}"
    
    result = call_llm_with_retry(system_prompt, user_message)
    
    if result is None:
        result = {"severity": "Medium", "confidence": 0.0, "reasoning": "LLM call failed after retries", "priority": "Medium", "component": "Unknown"}
    
    result['bug_id'] = bug_id
    return result

# Quick test
test_result = triage_agent(
    title="App crashes on login",
    description="Users report the app crashes immediately after entering valid credentials.",
    bug_id="TEST-001"
)
print(test_result)

{'bug_id': 'TEST-001', 'severity': 'Critical', 'confidence': 0.95, 'reasoning': 'The title explicitly states the app crashes, which is a clear indicator of a Critical severity issue. The description also confirms that the crash occurs after entering valid credentials, further supporting this classification.', 'priority': 'Immediate', 'component': 'Login'}


In [51]:
results_df.to_csv("../data/triage_validation_results.csv", index=False)
print("Saved validation results")

Saved validation results


# Task 2:Log Analysis Agent

In [2]:
import pandas as pd 
import glob

csv_files = glob.glob("../data/stack_traces/*.csv")
print("Found " , len(csv_files),"files :")
for f in csv_files:
    print("-",f)

Found  11 files :
- ../data/stack_traces\cpp_smart_analyzer_dataset_500.csv
- ../data/stack_traces\css_smart_analyzer_dataset_v2_500.csv
- ../data/stack_traces\c_smart_analyzer_dataset_500.csv
- ../data/stack_traces\go_smart_analyzer_dataset_500.csv
- ../data/stack_traces\html_smart_analyzer_dataset_v2_500.csv
- ../data/stack_traces\java_smart_analyzer_dataset_500.csv
- ../data/stack_traces\js_smart_analyzer_dataset_500.csv
- ../data/stack_traces\php_smart_analyzer_dataset_500.csv
- ../data/stack_traces\python_10_out_of_10_dataset_500.csv
- ../data/stack_traces\rust_smart_analyzer_dataset_v3_500.csv
- ../data/stack_traces\ts_smart_analyzer_dataset_500.csv


In [4]:
all_dfs = [pd.read_csv(f)  for f in csv_files]
stack_trace_df = pd.concat(all_dfs , ignore_index = True)
print("Combined shape :",stack_trace_df.shape)
print("\nLanguages distribution :")
print(stack_trace_df['language'].value_counts())

Combined shape : (5500, 7)

Languages distribution :
language
cpp           500
css           500
c             500
go            500
html          500
java          500
javascript    500
php           500
python        500
rust          500
typescript    500
Name: count, dtype: int64


In [5]:
stack_trace_df.to_csv("../data/stack_trace_combined.csv",index=False)
print("Saved combined stack trace dataset")

Saved combined stack trace dataset


In [6]:
" parses stack traces and error messages, identifies exception type, failure point, and affected code path with structured output."

' parses stack traces and error messages, identifies exception type, failure point, and affected code path with structured output.'

## Prompt 

In [9]:
'''Your job is extract the useful information for messy error text 
(like extracting  identifies exception type, failure point, and affected code path with structured output )

-  Input will be given to you  is  Raw Stack traces / messy error text / Traceback  or code 

-  What you have to do is :
    1) Cleans up the messy (Raw Stack traces / messy error text / Traceback  or code ) and you have to extract useful information like (like extracting  identifies exception type, failure point, and affected code path with structured output )

    2)Parses stack traces and error message  means converts raw, multi-line error logs into structured data objects containing individual frames with specific file names in jason format

     3) output should have these fields like :   
            error_type : what type of error is it , indexerror , syntaxerror , zerodivisionerror etc 
            
            failure_location, make it hold the actual line info from the traceback 
            
            code_path field describing the chain of function calls (even if it's short/simple for basic examples) 
            
            reasoning = a short sentence explaining why it concluded this. 
            
                Confidence here means something much simpler: "how sure am I that I read this correctly?".
                
                            What it actually means here: the Log Analysis Agent is just reading a messy error message and trying to extract facts from it (exception type, where it broke, etc.). 
                            
                            Sometimes the traceback is crystal clear, and sometimes it's messy/ambiguous. 
                            
                            "Confidence" is simply: "how certain am I that I extracted the right information from this specific error text?".
                            
                            if the stack trace is clean and obvious (like your IndexError example — very clear, exact line number given), confidence should be high (0.9+). 
                            
                            If the error text is vague or cut off, confidence would be lower — the AI is saying "I did my best to extract this, but the input wasn't fully clear."
                            
            

     
     examples for understanding:
1. Python
IndentationError

Input

Cell In[119], line 3
    return payload_2
    ^
IndentationError: unindent does not match any outer indentation level

Output

{
  "error_type": "IndentationError",
  "failure_location": "Cell In[119], line 3",
  "code_path": "return payload_2 executed directly within the current code block; indentation parsing failed before execution",
  "confidence": 0.99,
  "reasoning": "The 'return' statement has inconsistent indentation that does not align with the surrounding block, causing Python's parser to reject the code."
}

2. JavaScript
 TypeError

Input

/workspace/service_2.js:3
console.log(session_2[5].id);
                            ^

TypeError: Cannot read properties of undefined (reading 'id')
    at Object.<anonymous> (/workspace/service_2.js:3:15)
    at Module._compile (internal/modules/cjs/loader.js:1159:30)
    at Object.Module._extensions..js (internal/modules/cjs/loader.js:1195:10)

Output

{
  "error_type": "TypeError",
  "failure_location": "/workspace/service_2.js:3:15",
  "code_path": "Top-level execution → console.log(session_2[5].id)",
  "confidence": 0.99,
  "reasoning": "The expression 'session_2[5]' evaluated to undefined, so accessing its 'id' property caused a TypeError."
}

3. C
a. Invalid Initializer

Input

parser_4.c: In function ‘main’:
parser_4.c:3:15: error: invalid initializer / incompatible types when initializing type ‘int’ using type ‘char *’
    3 |     int val = str;
      |               ^~~

Output

{
  "error_type": "Compilation Error",
  "failure_location": "parser_4.c:3:15",
  "code_path": "main() → variable initialization: int val = str",
  "confidence": 0.85,
  "reasoning": "An integer variable was initialized using a character pointer, which is an incompatible type conversion in C."
}



4.C++
 No Matching Member Function

Input

session_2.cpp: In function ‘int main()’:
session_2.cpp:5:17: error: no matching member function for call to 'insert'
    5 |     ages.insert(10, "John");
      |     ~~~~~~~~~~~~^~~~~~~~~~~
In file included from /usr/include/c++/11/map:61:
/usr/include/c++/11/bits/stl_map.h:810:7: note: candidate function not viable: requires a single pair object

Output

{
  "error_type": "Compilation Error",
  "failure_location": "session_2.cpp:5:17",
  "code_path": "main() → std::map::insert(10, \"John\")",
  "confidence": 0.95,
  "reasoning": "The std::map::insert() function was called with arguments that do not match any valid overload. It expects a key-value pair or another supported insertion form."
}



5.Java
 NullPointerException

Input

Exception in thread "main" java.lang.NullPointerException: Cannot invoke "String.indexOf(String)" because "recordsList_2" is null
    at MetricsEngine_2.updateCache_2(Api_2.java:110)

Output

{
  "error_type": "NullPointerException",
  "failure_location": "Api_2.java:110",
  "code_path": "main thread → MetricsEngine_2.updateCache_2() → String.indexOf()",
  "confidence": 0.99,
  "reasoning": "The object 'recordsList_2' is null, so calling the 'indexOf()' method on it causes a NullPointerException."
}



'''

'Your job is extract the useful information for messy error text \n(like extracting  identifies exception type, failure point, and affected code path with structured output )\n\n-  Input will be given to you  is  Raw Stack traces / messy error text / Traceback  or code \n\n-  What you have to do is :\n    1) Cleans up the messy (Raw Stack traces / messy error text / Traceback  or code ) and you have to extract useful information like (like extracting  identifies exception type, failure point, and affected code path with structured output )\n\n    2)Parses stack traces and error message  means converts raw, multi-line error logs into structured data objects containing individual frames with specific file names in jason format\n\n     3) output should have these fields like :   \n     examples for understanding:\n1. Python\na. IndentationError\n\nInput\n\nCell In[119], line 3\n    return payload_2\n    ^\nIndentationError: unindent does not match any outer indentation level\n\nOutput\n\n

In [8]:
'''examples for understanding:
1. Python
a. IndentationError

Input

Cell In[119], line 3
    return payload_2
    ^
IndentationError: unindent does not match any outer indentation level

Output

{
  "error_type": "IndentationError",
  "failure_location": "Cell In[119], line 3",
  "code_path": "return payload_2 executed directly within the current code block; indentation parsing failed before execution",
  "confidence": 0.99,
  "reasoning": "The 'return' statement has inconsistent indentation that does not align with the surrounding block, causing Python's parser to reject the code."
}
b. NameError

Input

Traceback (most recent call last):
  Cell In[92], line 4, in <module>
    update_cache_4()
  Cell In[92], line 2, in update_cache_4
        1 def update_cache_4():
----> 2     return config_4_missing + 10

NameError: name 'config_4_missing' is not defined

Output

{
  "error_type": "NameError",
  "failure_location": "Cell In[92], line 2, in update_cache_4",
  "code_path": "<module> → update_cache_4() → return config_4_missing + 10",
  "confidence": 0.99,
  "reasoning": "The variable 'config_4_missing' was referenced inside the function before being defined or assigned."
}
2. JavaScript
a. TypeError

Input

/workspace/service_2.js:3
console.log(session_2[5].id);
                            ^

TypeError: Cannot read properties of undefined (reading 'id')
    at Object.<anonymous> (/workspace/service_2.js:3:15)
    at Module._compile (internal/modules/cjs/loader.js:1159:30)
    at Object.Module._extensions..js (internal/modules/cjs/loader.js:1195:10)

Output

{
  "error_type": "TypeError",
  "failure_location": "/workspace/service_2.js:3:15",
  "code_path": "Top-level execution → console.log(session_2[5].id)",
  "confidence": 0.99,
  "reasoning": "The expression 'session_2[5]' evaluated to undefined, so accessing its 'id' property caused a TypeError."
}
b. SyntaxError

Input

/workspace/service_6.js:1
const values = [1, 2, 3;
                       ^

SyntaxError: Unexpected token ';'

Output

{
  "error_type": "SyntaxError",
  "failure_location": "/workspace/service_6.js:1",
  "code_path": "Top-level variable declaration while parsing the JavaScript source",
  "confidence": 0.99,
  "reasoning": "The array literal was not closed with ']', so the parser encountered an unexpected semicolon."
}
3. C
a. Invalid Initializer

Input

parser_4.c: In function ‘main’:
parser_4.c:3:15: error: invalid initializer / incompatible types when initializing type ‘int’ using type ‘char *’
    3 |     int val = str;
      |               ^~~

Output

{
  "error_type": "Compilation Error",
  "failure_location": "parser_4.c:3:15",
  "code_path": "main() → variable initialization: int val = str",
  "confidence": 0.99,
  "reasoning": "An integer variable was initialized using a character pointer, which is an incompatible type conversion in C."
}
b. Double Free

Input

double free or corruption (out)
Aborted (core dumped)
(gdb) bt
#0  __GI_raise ...
#1  __GI_abort ...
#2  __libc_message ...
#3  malloc_printerr ...
#4  _int_free ...
#5  main () at matrix_12.c:5

Output

{
  "error_type": "Double Free / Memory Corruption",
  "failure_location": "matrix_12.c:5",
  "code_path": "main() → free() → _int_free() → malloc_printerr() → __libc_message() → __GI_abort()",
  "confidence": 0.98,
  "reasoning": "The program attempted to free the same memory block more than once (or corrupted heap metadata), causing the C runtime to abort execution."
}


4.CSS
a. CssSyntaxError

Input

CssSyntaxError: grid_2.css:20:1: Unclosed block
  18 |     background: #000;
  19 |     opacity: 0.9;
> 20 |
    | ^
  21 | .element-sibling_2 {

Output

{
  "error_type": "CssSyntaxError",
  "failure_location": "grid_2.css:20:1",
  "code_path": "CSS parser → grid_2.css → unclosed declaration block before '.element-sibling_2'",
  "confidence": 0.99,
  "reasoning": "A CSS block was not properly closed with '}', causing the parser to detect an unclosed block before the next selector."
}
b. Expected ';'

Input

Error: Expected ";".
  ╷
84 │     border-radius: 3px
  │                    ^
  ╵
  navigation_3.css 84:33  root stylesheet

Output

{
  "error_type": "CssSyntaxError",
  "failure_location": "navigation_3.css:84:33",
  "code_path": "CSS parser → root stylesheet → property declaration 'border-radius: 3px'",
  "confidence": 0.99,
  "reasoning": "The CSS declaration is missing a terminating semicolon, causing the parser to report a syntax error."
}
5.HTML
a. Stray End Tag

Input

Error: Stray end tag </span>.
From line 95, column 15; to line 95, column 22
item 1</p></span>\n</di

Output

{
  "error_type": "HTML Validation Error",
  "failure_location": "Line 95, columns 15-22",
  "code_path": "HTML parser → closing tags → unexpected </span>",
  "confidence": 0.99,
  "reasoning": "The closing </span> tag does not match any currently open <span> element, resulting in a stray end tag."
}
b. Missing alt Attribute

Input

Error: Element img is missing required attribute alt.
From line 45, column 1; to line 45, column 29
er_2.png"" class=""text-muted_2"">\n</bo

Output

{
  "error_type": "HTML Validation Error",
  "failure_location": "Line 45, columns 1-29",
  "code_path": "HTML parser → <img> element validation",
  "confidence": 0.99,
  "reasoning": "The <img> element is missing the required 'alt' attribute, which is required for accessibility and HTML validation."
}

6.C++
a. std::out_of_range

Input

terminate called after throwing an instance of 'std::out_of_range'
  what(): vector::_M_range_check: __n (which is 9) >= this->size() (which is 3)
Aborted (core dumped)
(gdb) bt
#0  __GI_raise
#1  __GI_abort
#2  __gnu_cxx::__verbose_terminate_handler()
#3  std::vector<int>::at(unsigned long)
#4  main() at session_1.cpp:5

Output

{
  "error_type": "std::out_of_range",
  "failure_location": "session_1.cpp:5",
  "code_path": "main() → std::vector<int>::at() → __gnu_cxx::__verbose_terminate_handler() → __GI_abort()",
  "confidence": 0.99,
  "reasoning": "The program attempted to access index 9 of a vector containing only 3 elements, causing std::vector::at() to throw a std::out_of_range exception."
}
b. No Matching Member Function

Input

session_2.cpp: In function ‘int main()’:
session_2.cpp:5:17: error: no matching member function for call to 'insert'
    5 |     ages.insert(10, "John");
      |     ~~~~~~~~~~~~^~~~~~~~~~~
In file included from /usr/include/c++/11/map:61:
/usr/include/c++/11/bits/stl_map.h:810:7: note: candidate function not viable: requires a single pair object

Output

{
  "error_type": "Compilation Error",
  "failure_location": "session_2.cpp:5:17",
  "code_path": "main() → std::map::insert(10, \"John\")",
  "confidence": 0.99,
  "reasoning": "The std::map::insert() function was called with arguments that do not match any valid overload. It expects a key-value pair or another supported insertion form."
}


7.TypeScript
a. TS2322 - Type Assignment Error

Input

app_1.ts:63:1 - error TS2322: Type 'string' is not assignable to type 'number'.

63 weightsTensor_1 = "invalid_assign_1";
  ~~~~~~~~~~~~~~~

Output

{
  "error_type": "TS2322",
  "failure_location": "app_1.ts:63:1",
  "code_path": "Top-level assignment → weightsTensor_1 = \"invalid_assign_1\"",
  "confidence": 0.99,
  "reasoning": "A string value was assigned to a variable that is declared with the type 'number', violating TypeScript's type checking rules."
}
b. TS2339 - Property Does Not Exist

Input

utils_2.ts:84:13 - error TS2339: Property 'value' does not exist on type 'object'.

84 console.log(metricsData_2.value);
                          ~~~~~

Output

{
  "error_type": "TS2339",
  "failure_location": "utils_2.ts:84:13",
  "code_path": "Top-level execution → console.log(metricsData_2.value)",
  "confidence": 0.99,
  "reasoning": "The code attempts to access the 'value' property on an object whose declared type does not define that property."
}
8.PHP
a. Parse Error

Input

PHP Parse error: syntax error, unexpected token "echo", expecting ";" in /var/www/html/index_1.php on line 64

Output

{
  "error_type": "PHP Parse Error",
  "failure_location": "/var/www/html/index_1.php:64",
  "code_path": "PHP parser → top-level script parsing near 'echo' statement",
  "confidence": 0.99,
  "reasoning": "The parser encountered an 'echo' statement where it expected a semicolon, indicating a missing statement terminator before this line."
}
b. Undefined Function

Input

PHP Fatal error: Uncaught Error: Call to undefined function json_ecnode_typo_$2() in /var/www/html/User_2.php:69
Stack trace:
#0 {main}
  thrown in /var/www/html/User_2.php on line 69

Output

{
  "error_type": "Call to Undefined Function",
  "failure_location": "/var/www/html/User_2.php:69",
  "code_path": "{main} → json_ecnode_typo_$2()",
  "confidence": 0.99,
  "reasoning": "The program attempted to call a function that is not defined or available in the current PHP environment."
}
9.Go
a. Runtime Panic - Index Out of Range

Input

panic: runtime error: index out of range [7] with length 3

goroutine 1 [running]:
main.main()
    /go/src/app/main_2.go:96 +0x3d

Output

{
  "error_type": "Runtime Panic",
  "failure_location": "/go/src/app/main_2.go:96",
  "code_path": "main.main() → slice/array index access",
  "confidence": 0.99,
  "reasoning": "The program attempted to access index 7 of a slice or array that contains only 3 elements, causing a runtime panic."
}
b. Unused Import

Input

./main_5.go:4:5: "math" imported and not used

Output

{
  "error_type": "Unused Import Error",
  "failure_location": "./main_5.go:4:5",
  "code_path": "Package import → import \"math\"",
  "confidence": 0.99,
  "reasoning": "The 'math' package was imported but never referenced in the program, which Go treats as a compilation error."
}

10.Rust
a. E0382 - Borrow of Moved Value

Input

error[E0382]: borrow of moved value: `cache_1`
 --> main_1.rs:99:20
  |
96 |     let cache_1 = String::from("data_3");
97 |     let config_alt_1 = cache_1;
99 |     println!("{}", cache_1);
  |                    ^ value borrowed here after move

Output

{
  "error_type": "E0382",
  "failure_location": "main_1.rs:99:20",
  "code_path": "main() → cache_1 moved to config_alt_1 → println!(cache_1)",
  "confidence": 0.99,
  "reasoning": "Ownership of 'cache_1' was moved to 'config_alt_1', so attempting to use 'cache_1' afterward violates Rust's ownership rules."
}
b. E0277 - Missing Debug Trait

Input

error[E0277]: `CustomStruct_3` doesn't implement `Debug`
 --> utils_3.rs:77:22
  |
77 |     println!("{:?}", instance_3);
  |                      ^ `CustomStruct_3` cannot be formatted using `{:?}` because it doesn't implement the trait

Output

{
  "error_type": "E0277",
  "failure_location": "utils_3.rs:77:22",
  "code_path": "println!(\"{:?}\", instance_3) → Debug trait check",
  "confidence": 0.99,
  "reasoning": "The code attempts to print 'instance_3' using the '{:?}' formatter, but its type does not implement the Debug trait required for formatted output."
}


11.Java
a. Cannot Find Symbol

Input

Api_1.java:29: error: cannot find symbol
        System.out.println(config_1_typo);
                           ^
  symbol:   variable config_1_typo
  location: class NetworkPacket_1

Output

{
  "error_type": "Cannot Find Symbol",
  "failure_location": "Api_1.java:29",
  "code_path": "NetworkPacket_1 → System.out.println(config_1_typo)",
  "confidence": 0.99,
  "reasoning": "The variable 'config_1_typo' is referenced but has not been declared or is not accessible within the current scope."
}
b. NullPointerException

Input

Exception in thread "main" java.lang.NullPointerException: Cannot invoke "String.indexOf(String)" because "recordsList_2" is null
    at MetricsEngine_2.updateCache_2(Api_2.java:110)

Output

{
  "error_type": "NullPointerException",
  "failure_location": "Api_2.java:110",
  "code_path": "main thread → MetricsEngine_2.updateCache_2() → String.indexOf()",
  "confidence": 0.99,
  "reasoning": "The object 'recordsList_2' is null, so calling the 'indexOf()' method on it causes a NullPointerException."
}'''

## Actual Prompt

In [ ]:
# log_analysis_system_prompt = """You are a Log Analysis Agent. Your job is to extract structured information from messy error text — stack traces, tracebacks, or compiler error messages, sometimes alongside the relevant code.

# Extract:
# - error_type: the SPECIFIC error/exception name as it would appear in real compiler/runtime output — NOT a generic category. For example, prefer "Template Mismatch" or "Dereferencing Void Pointer" over generic terms like "Compilation Error" or "Compilation Warning". Look for the specific technical term the error message itself uses (e.g., a specific error code, exception class name, or named violation), not a broad summary category.
# - failure_location: the exact file/line/position where the error occurred, taken directly from the trace
# - code_path: the chain of function/method calls leading to the failure (e.g., "main() -> processData() -> validateInput()"); for simple top-level errors with no call chain, describe what was directly executed
# - confidence: a score from 0 to 1 representing how certain you are that you extracted this correctly. Vary this realistically — very clear, unambiguous traces should score 0.95-0.99, while vague, truncated, or ambiguous input should score lower (0.5-0.8). Do not default to 0.99 for everything.
# - reasoning: a brief, specific explanation of why the error occurred

# Reply ONLY in valid JSON with this exact structure:
# {"error_type": "...", "failure_location": "...", "code_path": "...", "confidence": 0.9, "reasoning": "..."}

# Examples:

# Input: Cell In[119], line 3
#     return payload_2
#     ^
# IndentationError: unindent does not match any outer indentation level

# Output: {"error_type": "IndentationError", "failure_location": "Cell In[119], line 3", "code_path": "return payload_2 executed directly within the current code block; indentation parsing failed before execution", "confidence": 0.99, "reasoning": "The 'return' statement has inconsistent indentation that does not align with the surrounding block, causing Python's parser to reject the code."}

# Input: /workspace/service_2.js:3
# console.log(session_2[5].id);
# TypeError: Cannot read properties of undefined (reading 'id')
#     at Object.<anonymous> (/workspace/service_2.js:3:15)

# Output: {"error_type": "TypeError", "failure_location": "/workspace/service_2.js:3:15", "code_path": "Top-level execution -> console.log(session_2[5].id)", "confidence": 0.99, "reasoning": "The expression 'session_2[5]' evaluated to undefined, so accessing its 'id' property caused a TypeError."}

# Input: parser_4.c: In function 'main':
# parser_4.c:3:15: error: invalid initializer / incompatible types when initializing type 'int' using type 'char *'
#     3 |     int val = str;

# Output: {"error_type": "Invalid Initializer (Incompatible Types)", "failure_location": "parser_4.c:3:15", "code_path": "main() -> variable initialization: int val = str", "confidence": 0.85, "reasoning": "An integer variable was initialized using a character pointer, which is an incompatible type conversion in C."}

# Input: session_2.cpp: In function 'int main()':
# session_2.cpp:5:17: error: no matching member function for call to 'insert'
#     5 |     ages.insert(10, "John");

# Output: {"error_type": "No Matching Member Function", "failure_location": "session_2.cpp:5:17", "code_path": "main() -> std::map::insert(10, \\"John\\")", "confidence": 0.95, "reasoning": "The std::map::insert() function was called with arguments that do not match any valid overload."}

# Input: Exception in thread "main" java.lang.NullPointerException: Cannot invoke "String.indexOf(String)" because "recordsList_2" is null
#     at MetricsEngine_2.updateCache_2(Api_2.java:110)

# Output: {"error_type": "NullPointerException", "failure_location": "Api_2.java:110", "code_path": "main thread -> MetricsEngine_2.updateCache_2() -> String.indexOf()", "confidence": 0.99, "reasoning": "The object 'recordsList_2' is null, so calling the 'indexOf()' method on it causes a NullPointerException."}
# """

In [10]:
# log_analysis_system_prompt = """You are a Log Analysis Agent. Your job is to extract structured information from messy error text — stack traces, tracebacks, or compiler error messages, sometimes alongside the relevant code.

# Extract:
# - error_type: the specific error/exception name (e.g., IndexError, NullPointerException, TS2322)
# - failure_location: the exact file/line/position where the error occurred, taken directly from the trace
# - code_path: the chain of function/method calls leading to the failure (e.g., "main() -> processData() -> validateInput()"); for simple top-level errors with no call chain, describe what was directly executed
# - confidence: a score from 0 to 1 representing how certain you are that you extracted this correctly. Vary this realistically — very clear, unambiguous traces should score 0.95-0.99, while vague, truncated, or ambiguous input should score lower (0.5-0.8). Do not default to 0.99 for everything.
# - reasoning: a brief, specific explanation of why the error occurred

# Reply ONLY in valid JSON with this exact structure:
# {"error_type": "...", "failure_location": "...", "code_path": "...", "confidence": 0.9, "reasoning": "..."}

# Examples:

# Input: Cell In[119], line 3
#     return payload_2
#     ^
# IndentationError: unindent does not match any outer indentation level

# Output: {"error_type": "IndentationError", "failure_location": "Cell In[119], line 3", "code_path": "return payload_2 executed directly within the current code block; indentation parsing failed before execution", "confidence": 0.99, "reasoning": "The 'return' statement has inconsistent indentation that does not align with the surrounding block, causing Python's parser to reject the code."}

# Input: /workspace/service_2.js:3
# console.log(session_2[5].id);
# TypeError: Cannot read properties of undefined (reading 'id')
#     at Object.<anonymous> (/workspace/service_2.js:3:15)

# Output: {"error_type": "TypeError", "failure_location": "/workspace/service_2.js:3:15", "code_path": "Top-level execution -> console.log(session_2[5].id)", "confidence": 0.99, "reasoning": "The expression 'session_2[5]' evaluated to undefined, so accessing its 'id' property caused a TypeError."}

# Input: parser_4.c: In function 'main':
# parser_4.c:3:15: error: invalid initializer / incompatible types when initializing type 'int' using type 'char *'
#     3 |     int val = str;

# Output: {"error_type": "Compilation Error", "failure_location": "parser_4.c:3:15", "code_path": "main() -> variable initialization: int val = str", "confidence": 0.85, "reasoning": "An integer variable was initialized using a character pointer, which is an incompatible type conversion in C."}

# Input: session_2.cpp: In function 'int main()':
# session_2.cpp:5:17: error: no matching member function for call to 'insert'
#     5 |     ages.insert(10, "John");

# Output: {"error_type": "Compilation Error", "failure_location": "session_2.cpp:5:17", "code_path": "main() -> std::map::insert(10, \\"John\\")", "confidence": 0.95, "reasoning": "The std::map::insert() function was called with arguments that do not match any valid overload."}

# Input: Exception in thread "main" java.lang.NullPointerException: Cannot invoke "String.indexOf(String)" because "recordsList_2" is null
#     at MetricsEngine_2.updateCache_2(Api_2.java:110)

# Output: {"error_type": "NullPointerException", "failure_location": "Api_2.java:110", "code_path": "main thread -> MetricsEngine_2.updateCache_2() -> String.indexOf()", "confidence": 0.99, "reasoning": "The object 'recordsList_2' is null, so calling the 'indexOf()' method on it causes a NullPointerException."}
# """

# print("Prompt length:", len(log_analysis_system_prompt), "characters")

Prompt length: 3568 characters


In [41]:
log_analysis_system_prompt = """You are a Log Analysis Agent. Your job is to extract structured information from messy error text — stack traces, tracebacks, or compiler error messages, sometimes alongside the relevant code.

Extract:
- error_type: the SPECIFIC error/exception name as it would appear in real compiler/runtime output — NOT a generic category. For example, prefer "Template Mismatch" or "Dereferencing Void Pointer" over generic terms like "Compilation Error" or "Compilation Warning". Look for the specific technical term the error message itself uses, not a broad summary category.
- failure_location: the exact file/line/position where the error occurred, taken directly from the trace
- code_path: the chain of function/method calls leading to the failure. IMPORTANT: if the code being described contains double quote characters (e.g. from println!, print statements, or string literals), you MUST escape them as \\" so the JSON stays valid. Never include an unescaped " inside any field value.
- confidence: a score from 0 to 1 representing how certain you are that you extracted this correctly. Vary this realistically — very clear, unambiguous traces should score 0.95-0.99, while vague, truncated, or ambiguous input should score lower (0.5-0.8). Do not default to 0.99 for everything.
- reasoning: a brief, specific explanation of why the error occurred

Reply ONLY in valid JSON with this exact structure. Every field value must be a properly escaped JSON string — double quotes inside any value must be written as \\":
{"error_type": "...", "failure_location": "...", "code_path": "...", "confidence": 0.9, "reasoning": "..."}

Examples:

Input: Cell In[119], line 3
    return payload_2
    ^
IndentationError: unindent does not match any outer indentation level

Output: {"error_type": "IndentationError", "failure_location": "Cell In[119], line 3", "code_path": "return payload_2 executed directly within the current code block; indentation parsing failed before execution", "confidence": 0.99, "reasoning": "The 'return' statement has inconsistent indentation that does not align with the surrounding block, causing Python's parser to reject the code."}

Input: /workspace/service_2.js:3
console.log(session_2[5].id);
TypeError: Cannot read properties of undefined (reading 'id')
    at Object.<anonymous> (/workspace/service_2.js:3:15)

Output: {"error_type": "TypeError", "failure_location": "/workspace/service_2.js:3:15", "code_path": "Top-level execution -> console.log(session_2[5].id)", "confidence": 0.99, "reasoning": "The expression 'session_2[5]' evaluated to undefined, so accessing its 'id' property caused a TypeError."}

Input: parser_4.c: In function 'main':
parser_4.c:3:15: error: invalid initializer / incompatible types when initializing type 'int' using type 'char *'
    3 |     int val = str;

Output: {"error_type": "Invalid Initializer (Incompatible Types)", "failure_location": "parser_4.c:3:15", "code_path": "main() -> variable initialization: int val = str", "confidence": 0.85, "reasoning": "An integer variable was initialized using a character pointer, which is an incompatible type conversion in C."}

Input: session_2.cpp: In function 'int main()':
session_2.cpp:5:17: error: no matching member function for call to 'insert'
    5 |     ages.insert(10, "John");

Output: {"error_type": "No Matching Member Function", "failure_location": "session_2.cpp:5:17", "code_path": "main() -> std::map::insert(10, \\"John\\")", "confidence": 0.95, "reasoning": "The std::map::insert() function was called with arguments that do not match any valid overload."}

Input: Exception in thread "main" java.lang.NullPointerException: Cannot invoke "String.indexOf(String)" because "recordsList_2" is null
    at MetricsEngine_2.updateCache_2(Api_2.java:110)

Output: {"error_type": "NullPointerException", "failure_location": "Api_2.java:110", "code_path": "main thread -> MetricsEngine_2.updateCache_2() -> String.indexOf()", "confidence": 0.99, "reasoning": "The object 'recordsList_2' is null, so calling the 'indexOf()' method on it causes a NullPointerException."}

Input: CssSyntaxError: grid_2.css:20:1: Unclosed block
  18 |     background: #000;
  19 |     opacity: 0.9;
> 20 |
    | ^
  21 | .element-sibling_2 {

Output: {"error_type": "PostCSS Parser Error: Unclosed Block", "failure_location": "grid_2.css:20:1", "code_path": "CSS parser -> grid_2.css -> unclosed declaration block before '.element-sibling_2'", "confidence": 0.99, "reasoning": "A CSS block was not properly closed with '}', causing the parser to detect an unclosed block before the next selector."}

Input: Error: Stray end tag </span>.
From line 95, column 15; to line 95, column 22
item 1</p></span>\n</di

Output: {"error_type": "W3C Validator Error: Stray End Tag", "failure_location": "Line 95, columns 15-22", "code_path": "HTML parser -> closing tags -> unexpected </span>", "confidence": 0.99, "reasoning": "The closing </span> tag does not match any currently open <span> element, resulting in a stray end tag."}

Input: app_1.ts:63:1 - error TS2322: Type 'string' is not assignable to type 'number'.
63 weightsTensor_1 = "invalid_assign_1";
  ~~~~~~~~~~~~~~~

Output: {"error_type": "TypeScript Error: TS2322 (Type Assignment Error)", "failure_location": "app_1.ts:63:1", "code_path": "Top-level assignment -> weightsTensor_1 = \\"invalid_assign_1\\"", "confidence": 0.99, "reasoning": "A string value was assigned to a variable that is declared with the type 'number', violating TypeScript's type checking rules."}

Input: panic: runtime error: index out of range [7] with length 3
goroutine 1 [running]:
main.main()
    /go/src/app/main_2.go:96 +0x3d

Output: {"error_type": "Go Runtime Panic: Index Out of Range", "failure_location": "/go/src/app/main_2.go:96", "code_path": "main.main() -> slice/array index access", "confidence": 0.99, "reasoning": "The program attempted to access index 7 of a slice or array that contains only 3 elements, causing a runtime panic."}


"""

## validation

In [56]:
import time
def call_llm_with_retry(system_prompt, user_message, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_message}
                ],
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait_time = (attempt + 1) * 3
                print(f"Rate limited, waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"Non-rate-limit error: {e}")
                break
    return None

In [57]:
weak_test = validation_sample_large[validation_sample_large['language'].isin(['css', 'html', 'c', 'cpp'])]

for idx, row in weak_test.iterrows():
    predicted = log_analysis_agent(row['stack_trace'], code_context=row['code'])
    print(f"Language: {row['language']}")
    print(f"Actual: {row['error_type']}")
    print(f"Predicted: {predicted.get('error_type')}")
    print()

Language: cpp
Actual: g++ Compiler Error: Template Mismatch
Predicted: No Matching Member Function

Language: cpp
Actual: std::bad_alloc Exception
Predicted: std::bad_alloc

Language: cpp
Actual: g++ Compiler Warning: Safe Comparison Mismatch (Signed/Unsigned)
Predicted: Comparison Of Integer Expressions Of Different Signedness

Language: cpp
Actual: g++ Compiler Error: No Matching Constructor
Predicted: No Matching Function

Language: cpp
Actual: g++ Compiler Warning: Safe Comparison Mismatch (Signed/Unsigned)
Predicted: Comparison Of Integer Expressions Of Different Signedness

Language: css
Actual: PostCSS Parser Error: Unclosed Block
Predicted: CssSyntaxError

Language: css
Actual: Stylelint Error: Invalid Hex Color
Predicted: Unexpected invalid hex color

Language: css
Actual: Stylelint Error: Duplicate Property
Predicted: Unexpected duplicate "display" property

Language: css
Actual: Sass Compiler Error: Malformed calc()
Predicted: Expected Expression

Language: css
Actual: Style

In [58]:
import re

def is_match_v2(predicted, actual):
    def normalize(text):
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', '', text)  # remove punctuation like quotes, colons
        text = re.sub(r'\bs\b|s$', '', text)      # rough plural stripping (very simple)
        words = set(text.split())
        return words

    pred_words = normalize(predicted)
    actual_words = normalize(actual)

    # Remove common "noise" words that aren't the actual error concept
    noise_words = {'error', 'warning', 'exception', 'compiler', 'parser', 'validator', 'runtime'}
    pred_core = pred_words - noise_words
    actual_core = actual_words - noise_words

    # Check meaningful word overlap
    overlap = pred_core & actual_core
    return len(overlap) >= 2  # at least 2 shared meaningful words

In [60]:
def is_match_v3(predicted, actual):
    def normalize(text):
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', '', text)
        return set(text.split())

    pred_words = normalize(predicted)
    actual_words = normalize(actual)

    noise_words = {'error', 'warning', 'exception', 'compiler', 'parser', 'validator', 'runtime', 'gcc', 'g'}
    pred_core = pred_words - noise_words
    actual_core = actual_words - noise_words

    if len(pred_core) == 0 or len(actual_core) == 0:
        return False

    overlap = pred_core & actual_core
    # Require at least half of the SHORTER answer's words to match, minimum 1
    smaller_set_size = min(len(pred_core), len(actual_core))
    required = max(1, smaller_set_size // 2)
    return len(overlap) >= required

In [61]:
results_df_large['fair_match_v3'] = results_df_large.apply(
    lambda r: is_match_v3(r['predicted_error_type'], r['actual_error_type']), axis=1
)
fair_accuracy_v3 = results_df_large['fair_match_v3'].sum() / len(results_df_large) * 100
print(f"Fair match v3 accuracy: {results_df_large['fair_match_v3'].sum()}/{len(results_df_large)} = {fair_accuracy_v3:.1f}%")
print("\nPer-language fair match v3 accuracy:")
print(results_df_large.groupby('language')['fair_match_v3'].mean().sort_values() * 100)

Fair match v3 accuracy: 45/55 = 81.8%

Per-language fair match v3 accuracy:
language
cpp            40.0
typescript     60.0
css            80.0
c              80.0
javascript     80.0
go             80.0
rust           80.0
html          100.0
php           100.0
java          100.0
python        100.0
Name: fair_match_v3, dtype: float64


In [62]:
weak_test2 = validation_sample_large[validation_sample_large['language'].isin(['cpp', 'typescript'])]

spot_results = []
for idx, row in weak_test2.iterrows():
    predicted = log_analysis_agent(row['stack_trace'], code_context=row['code'])
    predicted_type = predicted.get('error_type', 'Unknown')
    actual_type = row['error_type']
    match = is_match_v3(predicted_type, actual_type)
    spot_results.append({'language': row['language'], 'actual': actual_type, 'predicted': predicted_type, 'match': match})
    print(f"{row['language']}: actual='{actual_type}' | predicted='{predicted_type}' | match={match}")

spot_df = pd.DataFrame(spot_results)
print(f"\nSpot-test accuracy: {spot_df['match'].sum()}/{len(spot_df)} = {spot_df['match'].mean()*100:.1f}%")

cpp: actual='g++ Compiler Error: Template Mismatch' | predicted='No Matching Member Function' | match=False
cpp: actual='std::bad_alloc Exception' | predicted='std::bad_alloc' | match=True
cpp: actual='g++ Compiler Warning: Safe Comparison Mismatch (Signed/Unsigned)' | predicted='Comparison Of Integer Expressions Of Different Signedness' | match=False
cpp: actual='g++ Compiler Error: No Matching Constructor' | predicted='No Matching Function' | match=True
cpp: actual='g++ Compiler Warning: Safe Comparison Mismatch (Signed/Unsigned)' | predicted='Comparison Of Integer Expressions Of Different Signedness' | match=False
typescript: actual='TypeScript Error: TS2339 (Property Does Not Exist)' | predicted='TS2339: Property 'value' does not exist on type 'object'' | match=True
typescript: actual='TypeScript Error: TS2564 (Uninitialized Property)' | predicted='TS2564' | match=True
typescript: actual='TypeScript Error: TS2531 (Object is Possibly Null)' | predicted='TS2531: Object is possibly 'n

## samples c

In [59]:
results_df_large['fair_match_v2'] = results_df_large.apply(
    lambda r: is_match_v2(r['predicted_error_type'], r['actual_error_type']), axis=1
)

fair_accuracy_v2 = results_df_large['fair_match_v2'].sum() / len(results_df_large) * 100
print(f"Fair match v2 accuracy: {results_df_large['fair_match_v2'].sum()}/{len(results_df_large)} = {fair_accuracy_v2:.1f}%")

print("\nPer-language fair match v2 accuracy:")
print(results_df_large.groupby('language')['fair_match_v2'].mean().sort_values() * 100)

Fair match v2 accuracy: 29/55 = 52.7%

Per-language fair match v2 accuracy:
language
javascript      0.0
python          0.0
cpp            20.0
java           40.0
php            40.0
c              60.0
typescript     60.0
css            80.0
rust           80.0
go            100.0
html          100.0
Name: fair_match_v2, dtype: float64


In [17]:
def log_analysis_agent(error_text, code_context=""):
    if code_context:
        user_message = f"Code:\n{code_context}\n\nError:\n{error_text}"
    else:
        user_message = f"Error:\n{error_text}"
    
    result = call_llm_with_retry(log_analysis_system_prompt, user_message)
    0
    if result is None:
        result = {
            "error_type": "Unknown",
            "failure_location": "Not determined",
            "code_path": "Not determined",
            "confidence": 0.0,
            "reasoning": "LLM call failed after retries"
        }
    
    return result

In [19]:
test_error = """Cell In[13], line 2
      1 loooo= [1,2]
----> 2 loooo[5]

IndexError: list index out of range"""

result = log_analysis_agent(test_error)
print(result)

{'error_type': 'IndexError', 'failure_location': 'Cell In[13], line 2', 'code_path': 'list indexing: loooo[5] executed directly within the current code block; list length exceeded index', 'confidence': 0.99, 'reasoning': "The list 'loooo' has a length of 2, which is less than the index 5, causing an IndexError when trying to access the element at that position."}


In [20]:
stack_trace_df = pd.read_csv("../data/stack_trace_combined.csv")

print("Shape :",stack_trace_df.shape)
print("\nColumns : ",stack_trace_df.columns.tolist())
print("\nSample row : ")
print(stack_trace_df.iloc[0])

Shape : (5500, 7)

Columns :  ['s.no', 'language', 'error_type', 'code', 'stack_trace', 'fix_brief', 'fix_solution']

Sample row : 
s.no                                                            1
language                                                      cpp
error_type                            std::out_of_range Exception
code            #include <vector>\n#include <iostream>\nint ma...
stack_trace     terminate called after throwing an instance of...
fix_brief       Ensure requested indexes sit cleanly inside th...
fix_solution    #include <vector>\n#include <iostream>\nint ma...
Name: 0, dtype: object


In [ ]:
validation_sample = stack_trace_df.groupby('language',group_keys = False).apply(
    lambda x:x.sample(3,random_state = 42).reset_index(drop=True))

print("Validation sample size :",len(validation_sample))
print('\nPer-language breakdown : ')
print(validation_sample["language"].value_counts())


In [28]:
languages = stack_trace_df['language'].unique()

validation_sample = pd.concat([
    stack_trace_df[stack_trace_df['language'] == lang].sample(3,random_state=42)
    for lang in languages 
]).reset_index(drop = True)

print("Validation sample size : ",len(validation_sample))
print("\nPer-language breakdown :")
print(validation_sample['language'].value_counts())

Validation sample size :  33

Per-language breakdown :
language
cpp           3
css           3
c             3
go            3
html          3
java          3
javascript    3
php           3
python        3
rust          3
typescript    3
Name: count, dtype: int64


In [38]:
import time

exact_matches = 0
results_log = []

for idx,row in validation_sample.iterrows():
    predicted = log_analysis_agent(row['stack_trace'],code_context = row['code'])
    predicted_type = predicted.get('error_type','Unknown')
    actual_type = row['error_type']

    is_exact = (predicted_type.strip().lower()== actual_type.strip().lower())
    if is_exact:
        exact_matches += 1 
    results_log.append({
        'language':row['language'],
        'actual_error_type': actual_type,
        'predicted_error_type': predicted_type,
        'confidence':predicted.get('confidence' , None),
        'exact_match':is_exact
        
    })
    time.sleep(2)

print(f"\nExact match accuracy : {exact_matches}/{len(validation_sample)} = {exact_matches/len(validation_sample)*100:.1f}%")
    

Non-rate-limit error: Expecting ',' delimiter: line 1 column 116 (char 115)

Exact match accuracy : 5/33 = 15.2%


In [34]:
results_df = pd.DataFrame(results_log)
pd.set_option('display.max_colwidth', 60)
print(results_df[['language', 'actual_error_type', 'predicted_error_type', 'exact_match']].to_string())

      language                                                 actual_error_type                                   predicted_error_type  exact_match
0          cpp                             g++ Compiler Error: Template Mismatch                                      Compilation Error        False
1          cpp                                          std::bad_alloc Exception                                         std::bad_alloc        False
2          cpp  g++ Compiler Warning: Safe Comparison Mismatch (Signed/Unsigned)                                                Warning        False
3          css                              PostCSS Parser Error: Unclosed Block                                         CssSyntaxError        False
4          css                                Stylelint Error: Invalid Hex Color                                      Invalid Hex Color        False
5          css                               Stylelint Error: Duplicate Property                         U

In [39]:
results_df = pd.DataFrame(results_log)

def is_match(predicted, actual):
    predicted_clean = predicted.strip().lower()
    actual_clean = actual.strip().lower()
    return predicted_clean in actual_clean or actual_clean in predicted_clean

results_df['fair_match'] = results_df.apply(
    lambda r: is_match(r['predicted_error_type'], r['actual_error_type']), axis=1
)

fair_accuracy = results_df['fair_match'].sum() / len(results_df) * 100
print(f"Fair match accuracy: {results_df['fair_match'].sum()}/{len(results_df)} = {fair_accuracy:.1f}%")

Fair match accuracy: 18/33 = 54.5%


In [36]:
print(results_df[results_df['fair_match'] == False][['language', 'actual_error_type', 'predicted_error_type']].to_string())

   language                                      actual_error_type                                   predicted_error_type
0       cpp                  g++ Compiler Error: Template Mismatch                                      Compilation Error
3       css                   PostCSS Parser Error: Unclosed Block                                         CssSyntaxError
5       css                    Stylelint Error: Duplicate Property                         Unexpected duplicate "display"
6         c                       GCC Compiler Error: Syntax Error                                      Compilation Error
7         c         GCC Compiler Error: Dereferencing Void Pointer                                      Compilation Error
8         c          GCC Compiler Error: Incompatible Pointer Type                                    Compilation Warning
9        go                   Go Runtime Panic: Index Out of Range               panic: runtime error: index out of range
10       go             

In [42]:
row_29 = validation_sample.loc[29]

call_llm_debug(log_analysis_system_prompt, f"Code:\n{row_29['code']}\n\nError:\n{row_29['stack_trace']}")

RAW RESPONSE:
{"error_type": "Index Out of Bounds", "failure_location": "config_375.rs:36:20", "code_path": "main() -> println!(\"{}\", profile_375[378])", "confidence": 0.99, "reasoning": "The vector 'profile_375' has a length of 3, but the index 378 is out of bounds, as indices must be less than the vector's length."}

--- Attempting to parse ---
SUCCESS: {'error_type': 'Index Out of Bounds', 'failure_location': 'config_375.rs:36:20', 'code_path': 'main() -> println!("{}", profile_375[378])', 'confidence': 0.99, 'reasoning': "The vector 'profile_375' has a length of 3, but the index 378 is out of bounds, as indices must be less than the vector's length."}


'{"error_type": "Index Out of Bounds", "failure_location": "config_375.rs:36:20", "code_path": "main() -> println!(\\"{}\\", profile_375[378])", "confidence": 0.99, "reasoning": "The vector \'profile_375\' has a length of 3, but the index 378 is out of bounds, as indices must be less than the vector\'s length."}'

In [50]:
languages = stack_trace_df['language'].unique()

validation_sample_large = pd.concat([
    stack_trace_df[stack_trace_df['language'] == lang].sample(5, random_state=42)
    for lang in languages
]).reset_index(drop=True)

print("Validation sample size:", len(validation_sample_large))
print(validation_sample_large['language'].value_counts())

Validation sample size: 55
language
cpp           5
css           5
c             5
go            5
html          5
java          5
javascript    5
php           5
python        5
rust          5
typescript    5
Name: count, dtype: int64


In [51]:
import time

exact_matches = 0
fair_matches = 0
results_log_large = []

def is_match(predicted, actual):
    predicted_clean = predicted.strip().lower()
    actual_clean = actual.strip().lower()
    return predicted_clean in actual_clean or actual_clean in predicted_clean

for idx, row in validation_sample_large.iterrows():
    predicted = log_analysis_agent(row['stack_trace'], code_context=row['code'])
    predicted_type = predicted.get('error_type', 'Unknown')
    actual_type = row['error_type']

    is_exact = (predicted_type.strip().lower() == actual_type.strip().lower())
    is_fair = is_match(predicted_type, actual_type)

    if is_exact:
        exact_matches += 1
    if is_fair:
        fair_matches += 1

    results_log_large.append({
        'language': row['language'],
        'actual_error_type': actual_type,
        'predicted_error_type': predicted_type,
        'confidence': predicted.get('confidence', None),
        'exact_match': is_exact,
        'fair_match': is_fair
    })

    if idx % 20 == 0:
        print(f"Progress: {idx}/{len(validation_sample_large)}")

    time.sleep(1.5)

print(f"\nExact match accuracy: {exact_matches}/{len(validation_sample_large)} = {exact_matches/len(validation_sample_large)*100:.1f}%")
print(f"Fair match accuracy: {fair_matches}/{len(validation_sample_large)} = {fair_matches/len(validation_sample_large)*100:.1f}%")

Progress: 0/55
Progress: 20/55
Progress: 40/55
Non-rate-limit error: Expecting ',' delimiter: line 1 column 219 (char 218)
Non-rate-limit error: Expecting ',' delimiter: line 1 column 147 (char 146)

Exact match accuracy: 5/55 = 9.1%
Fair match accuracy: 32/55 = 58.2%


In [52]:
results_df_large = pd.DataFrame(results_log_large)

print("Per-language fair match accuracy:")
print(results_df_large.groupby('language')['fair_match'].mean().sort_values() * 100)

Per-language fair match accuracy:
language
typescript      0.0
c              20.0
css            20.0
cpp            40.0
html           40.0
go             60.0
rust           80.0
javascript     80.0
php           100.0
java          100.0
python        100.0
Name: fair_match, dtype: float64


In [47]:
results_df_large = pd.DataFrame(results_log_large)

print("Per-language fair match accuracy:")
print(results_df_large.groupby('language')['fair_match'].mean().sort_values() * 100)

Per-language fair match accuracy:
language
css            10.0
typescript     10.0
html           20.0
c              20.0
cpp            50.0
rust           60.0
go             70.0
java           90.0
php            90.0
javascript     90.0
python        100.0
Name: fair_match, dtype: float64


In [48]:
css_fails = results_df_large[(results_df_large['language'] == 'css') & (results_df_large['fair_match'] == False)]
print(css_fails[['actual_error_type', 'predicted_error_type']].to_string())

                            actual_error_type                     predicted_error_type
11         Stylelint Error: Invalid Hex Color             Unexpected invalid hex color
12        Stylelint Error: Duplicate Property           Unexpected duplicate "display"
13      Sass Compiler Error: Malformed calc()                      Expected Expression
14        Stylelint Error: Duplicate Property  Unexpected duplicate "display" property
15        Stylelint Error: Duplicate Property  Unexpected duplicate "display" property
16        PostCSS Parser Error: Missing Colon                             Missed Colon
17        Stylelint Error: Duplicate Property  Unexpected duplicate "display" property
18  DevTools CSS Warning: Invalid Media Query         Invalid Media Query Length Value
19          Stylelint Error: Unknown Property              Unexpected unknown property


In [63]:
results_df_large.to_csv("../data/log_analysis_validation_results.csv", index=False)
print("Saved")

Saved


#  Task 3 : Combined Result 

In [15]:
def run_orchestration(title,description , stack_trace , bug_id = "unknown"):
    triage_result = triage_agent(title , description , stack_trace , bug_id=bug_id)
    log_result = log_analysis_agent(stack_trace)
    combined_result= {
        "bug_id" : bug_id,
        "triage" : triage_result,
        "log_analysis" : log_result
    }

    return combined_result

def build_simple_view (combined_result):
    triage = combined_result['triage']
    log = combined_result['log_analysis']

    actual_result = {
        'bug_id':combined_result['bug_id'],
        'severity':triage['severity'],
        'priority':triage['priority'],
        'component' : triage['component'],
        'error_type':log['error_type'],
        'failure_location':log['failure_location'],
        'code_path':log['code_path'],
        'confidence':log['confidence'],
        'reasoning':log['reasoning']
    }

    return actual_result

In [19]:
result = run_orchestration(test_title,test_description , test_stack_trace ,bug_id = "TEST-001")
print("Combined Result (save , sent to Milestone 3 ) :")
print(result)
 

print()

simple = build_simple_view(result)
print("Simple View (for UI default disply) ")
print(simple)

Combined Result (save , sent to Milestone 3 ) :
{'bug_id': 'TEST-001', 'triage': {'bug_id': 'TEST-001', 'severity': 'Critical', 'confidence': 0.9, 'reasoning': 'The app crashes immediately after a critical action (login) with a NullPointerException, indicating a significant issue that completely blocks core functionality.', 'priority': 'Immediate', 'component': 'Login'}, 'log_analysis': {'error_type': 'NullPointerException', 'failure_location': 'Api_2.java:110', 'code_path': 'main thread -> MetricsEngine_2.updateCache_2() -> String.indexOf()', 'confidence': 0.99, 'reasoning': "The object 'recordsList_2' is null, so calling the 'indexOf()' method on it causes a NullPointerException."}}

Simple View (for UI default disply) 
{'bug_id': 'TEST-001', 'severity': 'Critical', 'priority': 'Immediate', 'component': 'Login', 'error_type': 'NullPointerException', 'failure_location': 'Api_2.java:110', 'code_path': 'main thread -> MetricsEngine_2.updateCache_2() -> String.indexOf()', 'confidence': 0

# ERROR MAKING AND TESTING 

In [53]:
import json

def call_llm_debug(system_prompt, user_message):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0.2
    )
    raw = response.choices[0].message.content
    print("RAW RESPONSE:")
    print(raw)
    print("\n--- Attempting to parse ---")
    try:
        parsed = json.loads(raw)
        print("SUCCESS:", parsed)
    except Exception as e:
        print("FAILED:", e)
    return raw

# Test on the Rust rows specifically, since those failed before
rust_rows = validation_sample[validation_sample['language'] == 'rust']

for idx, row in rust_rows.iterrows():
    print(f"\n=== Row {idx} ===")
    call_llm_debug(log_analysis_system_prompt, f"Code:\n{row['code']}\n\nError:\n{row['stack_trace']}")
    print("=" * 50)


=== Row 27 ===
RAW RESPONSE:
{"error_type": "Cannot Borrow As Mutable Because It Is Also Borrowed As Immutable", "failure_location": "config_362.rs:23:5", "code_path": "main() -> let r_362 = &records_362[0]; -> records_362.push(365)", "confidence": 0.99, "reasoning": "The vector 'records_362' is immutably borrowed when 'r_362' is created, and this borrow is still active when 'records_362.push(365)' is called, which attempts a mutable borrow, causing a conflict."}

--- Attempting to parse ---
SUCCESS: {'error_type': 'Cannot Borrow As Mutable Because It Is Also Borrowed As Immutable', 'failure_location': 'config_362.rs:23:5', 'code_path': 'main() -> let r_362 = &records_362[0]; -> records_362.push(365)', 'confidence': 0.99, 'reasoning': "The vector 'records_362' is immutably borrowed when 'r_362' is created, and this borrow is still active when 'records_362.push(365)' is called, which attempts a mutable borrow, causing a conflict."}

=== Row 28 ===
RAW RESPONSE:
{"error_type": "Mismatch

In [13]:
list[]

IndexError: list index out of range

In [ ]:
'''1. Python

a. NameError

Input

Traceback (most recent call last):
  Cell In[92], line 4, in <module>
    update_cache_4()
  Cell In[92], line 2, in update_cache_4
        1 def update_cache_4():
----> 2     return config_4_missing + 10

NameError: name 'config_4_missing' is not defined

Output

{
  "error_type": "NameError",
  "failure_location": "Cell In[92], line 2, in update_cache_4",
  "code_path": "<module> → update_cache_4() → return config_4_missing + 10",
  "confidence": 0.99,
  "reasoning": "The variable 'config_4_missing' was referenced inside the function before being defined or assigned."
}
2. JavaScript
a. SyntaxError

Input

/workspace/service_6.js:1
const values = [1, 2, 3;
                       ^

SyntaxError: Unexpected token ';'

Output

{
  "error_type": "SyntaxError",
  "failure_location": "/workspace/service_6.js:1",
  "code_path": "Top-level variable declaration while parsing the JavaScript source",
  "confidence": 0.99,
  "reasoning": "The array literal was not closed with ']', so the parser encountered an unexpected semicolon."
}
3. C
a. Double Free

Input

double free or corruption (out)
Aborted (core dumped)
(gdb) bt
#0  __GI_raise ...
#1  __GI_abort ...
#2  __libc_message ...
#3  malloc_printerr ...
#4  _int_free ...
#5  main () at matrix_12.c:5

Output

{
  "error_type": "Double Free / Memory Corruption",
  "failure_location": "matrix_12.c:5",
  "code_path": "main() → free() → _int_free() → malloc_printerr() → __libc_message() → __GI_abort()",
  "confidence": 0.98,
  "reasoning": "The program attempted to free the same memory block more than once (or corrupted heap metadata), causing the C runtime to abort execution."
}


4.CSS
a. CssSyntaxError

Input

CssSyntaxError: grid_2.css:20:1: Unclosed block
  18 |     background: #000;
  19 |     opacity: 0.9;
> 20 |
    | ^
  21 | .element-sibling_2 {

Output

{
  "error_type": "CssSyntaxError",
  "failure_location": "grid_2.css:20:1",
  "code_path": "CSS parser → grid_2.css → unclosed declaration block before '.element-sibling_2'",
  "confidence": 0.99,
  "reasoning": "A CSS block was not properly closed with '}', causing the parser to detect an unclosed block before the next selector."
}
b. Expected ';'

Input

Error: Expected ";".
  ╷
84 │     border-radius: 3px
  │                    ^
  ╵
  navigation_3.css 84:33  root stylesheet

Output

{
  "error_type": "CssSyntaxError",
  "failure_location": "navigation_3.css:84:33",
  "code_path": "CSS parser → root stylesheet → property declaration 'border-radius: 3px'",
  "confidence": 0.99,
  "reasoning": "The CSS declaration is missing a terminating semicolon, causing the parser to report a syntax error."
}
5.HTML
a. Stray End Tag

Input

Error: Stray end tag </span>.
From line 95, column 15; to line 95, column 22
item 1</p></span>\n</di

Output

{
  "error_type": "HTML Validation Error",
  "failure_location": "Line 95, columns 15-22",
  "code_path": "HTML parser → closing tags → unexpected </span>",
  "confidence": 0.99,
  "reasoning": "The closing </span> tag does not match any currently open <span> element, resulting in a stray end tag."
}
b. Missing alt Attribute

Input

Error: Element img is missing required attribute alt.
From line 45, column 1; to line 45, column 29
er_2.png"" class=""text-muted_2"">\n</bo

Output

{
  "error_type": "HTML Validation Error",
  "failure_location": "Line 45, columns 1-29",
  "code_path": "HTML parser → <img> element validation",
  "confidence": 0.99,
  "reasoning": "The <img> element is missing the required 'alt' attribute, which is required for accessibility and HTML validation."
}

6.C++
a. std::out_of_range

Input

terminate called after throwing an instance of 'std::out_of_range'
  what(): vector::_M_range_check: __n (which is 9) >= this->size() (which is 3)
Aborted (core dumped)
(gdb) bt
#0  __GI_raise
#1  __GI_abort
#2  __gnu_cxx::__verbose_terminate_handler()
#3  std::vector<int>::at(unsigned long)
#4  main() at session_1.cpp:5

Output

{
  "error_type": "std::out_of_range",
  "failure_location": "session_1.cpp:5",
  "code_path": "main() → std::vector<int>::at() → __gnu_cxx::__verbose_terminate_handler() → __GI_abort()",
  "confidence": 0.99,
  "reasoning": "The program attempted to access index 9 of a vector containing only 3 elements, causing std::vector::at() to throw a std::out_of_range exception."
}



7.TypeScript
a. TS2322 - Type Assignment Error

Input

app_1.ts:63:1 - error TS2322: Type 'string' is not assignable to type 'number'.

63 weightsTensor_1 = "invalid_assign_1";
  ~~~~~~~~~~~~~~~

Output

{
  "error_type": "TS2322",
  "failure_location": "app_1.ts:63:1",
  "code_path": "Top-level assignment → weightsTensor_1 = \"invalid_assign_1\"",
  "confidence": 0.99,
  "reasoning": "A string value was assigned to a variable that is declared with the type 'number', violating TypeScript's type checking rules."
}
b. TS2339 - Property Does Not Exist

Input

utils_2.ts:84:13 - error TS2339: Property 'value' does not exist on type 'object'.

84 console.log(metricsData_2.value);
                          ~~~~~

Output

{
  "error_type": "TS2339",
  "failure_location": "utils_2.ts:84:13",
  "code_path": "Top-level execution → console.log(metricsData_2.value)",
  "confidence": 0.99,
  "reasoning": "The code attempts to access the 'value' property on an object whose declared type does not define that property."
}
8.PHP
a. Parse Error

Input

PHP Parse error: syntax error, unexpected token "echo", expecting ";" in /var/www/html/index_1.php on line 64

Output

{
  "error_type": "PHP Parse Error",
  "failure_location": "/var/www/html/index_1.php:64",
  "code_path": "PHP parser → top-level script parsing near 'echo' statement",
  "confidence": 0.99,
  "reasoning": "The parser encountered an 'echo' statement where it expected a semicolon, indicating a missing statement terminator before this line."
}
b. Undefined Function

Input

PHP Fatal error: Uncaught Error: Call to undefined function json_ecnode_typo_$2() in /var/www/html/User_2.php:69
Stack trace:
#0 {main}
  thrown in /var/www/html/User_2.php on line 69

Output

{
  "error_type": "Call to Undefined Function",
  "failure_location": "/var/www/html/User_2.php:69",
  "code_path": "{main} → json_ecnode_typo_$2()",
  "confidence": 0.99,
  "reasoning": "The program attempted to call a function that is not defined or available in the current PHP environment."
}
9.Go
a. Runtime Panic - Index Out of Range

Input

panic: runtime error: index out of range [7] with length 3

goroutine 1 [running]:
main.main()
    /go/src/app/main_2.go:96 +0x3d

Output

{
  "error_type": "Runtime Panic",
  "failure_location": "/go/src/app/main_2.go:96",
  "code_path": "main.main() → slice/array index access",
  "confidence": 0.99,
  "reasoning": "The program attempted to access index 7 of a slice or array that contains only 3 elements, causing a runtime panic."
}
b. Unused Import

Input

./main_5.go:4:5: "math" imported and not used

Output

{
  "error_type": "Unused Import Error",
  "failure_location": "./main_5.go:4:5",
  "code_path": "Package import → import \"math\"",
  "confidence": 0.99,
  "reasoning": "The 'math' package was imported but never referenced in the program, which Go treats as a compilation error."
}

10.Rust
a. E0382 - Borrow of Moved Value

Input

error[E0382]: borrow of moved value: `cache_1`
 --> main_1.rs:99:20
  |
96 |     let cache_1 = String::from("data_3");
97 |     let config_alt_1 = cache_1;
99 |     println!("{}", cache_1);
  |                    ^ value borrowed here after move

Output

{
  "error_type": "E0382",
  "failure_location": "main_1.rs:99:20",
  "code_path": "main() → cache_1 moved to config_alt_1 → println!(cache_1)",
  "confidence": 0.99,
  "reasoning": "Ownership of 'cache_1' was moved to 'config_alt_1', so attempting to use 'cache_1' afterward violates Rust's ownership rules."
}
b. E0277 - Missing Debug Trait

Input

error[E0277]: `CustomStruct_3` doesn't implement `Debug`
 --> utils_3.rs:77:22
  |
77 |     println!("{:?}", instance_3);
  |                      ^ `CustomStruct_3` cannot be formatted using `{:?}` because it doesn't implement the trait

Output

{
  "error_type": "E0277",
  "failure_location": "utils_3.rs:77:22",
  "code_path": "println!(\"{:?}\", instance_3) → Debug trait check",
  "confidence": 0.99,
  "reasoning": "The code attempts to print 'instance_3' using the '{:?}' formatter, but its type does not implement the Debug trait required for formatted output."
}


11.Java
a. Cannot Find Symbol

Input

Api_1.java:29: error: cannot find symbol
        System.out.println(config_1_typo);
                           ^
  symbol:   variable config_1_typo
  location: class NetworkPacket_1

Output

{
  "error_type": "Cannot Find Symbol",
  "failure_location": "Api_1.java:29",
  "code_path": "NetworkPacket_1 → System.out.println(config_1_typo)",
  "confidence": 0.99,
  "reasoning": "The variable 'config_1_typo' is referenced but has not been declared or is not accessible within the current scope."
}



'''

# Re-Run Codes

In [ ]:
import os
print("Current working directory:", os.getcwd())
print("Looking for .env at ../.env — exists:", os.path.exists("../.env"))

In [3]:
if os.path.exists("../.env"):
    with open("../.env", "r") as f:
        content = f.read()
    print("File has content:", len(content) > 0)
    print("Contains 'GROQ_API_KEY':", "GROQ_API_KEY" in content)

File has content: True
Contains 'GROQ_API_KEY': False


In [4]:
with open("../.env", "r") as f:
    content = f.read()

print("Content length:", len(content))
print("First 20 characters:", content[:20])
print("Number of lines:", len(content.splitlines()))

Content length: 77
First 20 characters: bug-analyzer-triage=
Number of lines: 1


In [5]:
env_content = "GROQ_API_KEY=API_key_value"

with open("../.env", "w") as f:
    f.write(env_content)

print("Rewrote .env file")

Rewrote .env file


In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env", override=True)
api_key = os.getenv("GROQ_API_KEY")
print("API key loaded:", api_key[:8] if api_key else "NOT FOUND")

In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env", override=True)
api_key = os.getenv("GROQ_API_KEY")
print("API key loaded:", api_key[:8] if api_key else "NOT FOUND")

In [8]:
client = Groq(api_key=api_key)
print("Client created successfully")

Client created successfully


In [9]:
def call_llm_with_retry(system_prompt, user_message, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_message}
                ],
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait_time = (attempt + 1) * 3
                print(f"Rate limited, waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"Non-rate-limit error: {e}")
                break
    return None

In [11]:
system_prompt = """You are a bug triage assistant. You are given the title, description, and stack trace of a submitted bug, and you classify it by severity, priority, and affected component, with confidence scoring and reasoning.

IMPORTANT CALIBRATION: Most everyday bugs are Medium severity, not Critical or High. Reserve Critical/High for bugs that cause crashes, data loss, security issues, or completely block core functionality. Reserve Low for cosmetic issues, minor inconveniences, or feature requests. The MAJORITY of real-world bugs are routine Medium severity - do not over-classify ordinary bugs as urgent just because they mention words like "error" or "exception."

Examples for calibration:
- "App crashes on startup, no workaround" -> Critical
- "Login fails for all users" -> Critical  
- "Button text is misaligned by 2px" -> Low
- "Feature request: add dark mode" -> Low
- "Export function occasionally produces incorrect date format" -> Medium
- "Search results are slow to load under heavy traffic" -> Medium

Reply ONLY in valid JSON, with this exact structure:
{"bug_id": "...", "severity": "Critical", "confidence": 0.85, "reasoning": "...", "priority": "...", "component": "Login"}

Field definitions:
- bug_id: if provided, use it; otherwise use "unknown"
- severity: must be exactly one of Critical, High, Medium, Low - no other words
- confidence: a score from 0 to 1 representing how confident you are in this classification
- reasoning: a brief explanation of why you classified it this way
- priority: how urgently this needs to be addressed - Immediate, High, Medium, or Low (related to but different from severity: severity is impact, priority is urgency)
- component: which part/module of the software seems affected, inferred from the content (e.g. "Login", "Database", "UI/Rendering") - do not restrict to a fixed list

If you encounter other severity-like words, map them using this table:
blocker/critical -> Critical, major -> High, normal/unknown -> Medium, minor/trivial -> Low
"""

log_analysis_system_prompt = """You are a Log Analysis Agent. Your job is to extract structured information from messy error text - stack traces, tracebacks, or compiler error messages, sometimes alongside the relevant code.

Extract:
- error_type: the SPECIFIC error/exception name as it would appear in real compiler/runtime output - NOT a generic category. For example, prefer "Template Mismatch" or "Dereferencing Void Pointer" over generic terms like "Compilation Error" or "Compilation Warning". Look for the specific technical term the error message itself uses, not a broad summary category.
- failure_location: the exact file/line/position where the error occurred, taken directly from the trace
- code_path: the chain of function/method calls leading to the failure. IMPORTANT: if the code being described contains double quote characters (e.g. from println!, print statements, or string literals), you MUST escape them as \\" so the JSON stays valid. Never include an unescaped " inside any field value.
- confidence: a score from 0 to 1 representing how certain you are that you extracted this correctly. Vary this realistically - very clear, unambiguous traces should score 0.95-0.99, while vague, truncated, or ambiguous input should score lower (0.5-0.8). Do not default to 0.99 for everything.
- reasoning: a brief, specific explanation of why the error occurred

Reply ONLY in valid JSON with this exact structure. Every field value must be a properly escaped JSON string - double quotes inside any value must be written as \\":
{"error_type": "...", "failure_location": "...", "code_path": "...", "confidence": 0.9, "reasoning": "..."}

Examples:

Input: Cell In[119], line 3
    return payload_2
    ^
IndentationError: unindent does not match any outer indentation level

Output: {"error_type": "IndentationError", "failure_location": "Cell In[119], line 3", "code_path": "return payload_2 executed directly within the current code block; indentation parsing failed before execution", "confidence": 0.99, "reasoning": "The 'return' statement has inconsistent indentation that does not align with the surrounding block, causing Python's parser to reject the code."}

Input: /workspace/service_2.js:3
console.log(session_2[5].id);
TypeError: Cannot read properties of undefined (reading 'id')
    at Object.<anonymous> (/workspace/service_2.js:3:15)

Output: {"error_type": "TypeError", "failure_location": "/workspace/service_2.js:3:15", "code_path": "Top-level execution -> console.log(session_2[5].id)", "confidence": 0.99, "reasoning": "The expression 'session_2[5]' evaluated to undefined, so accessing its 'id' property caused a TypeError."}

Input: parser_4.c: In function 'main':
parser_4.c:3:15: error: invalid initializer / incompatible types when initializing type 'int' using type 'char *'
    3 |     int val = str;

Output: {"error_type": "Invalid Initializer (Incompatible Types)", "failure_location": "parser_4.c:3:15", "code_path": "main() -> variable initialization: int val = str", "confidence": 0.85, "reasoning": "An integer variable was initialized using a character pointer, which is an incompatible type conversion in C."}

Input: session_2.cpp: In function 'int main()':
session_2.cpp:5:17: error: no matching member function for call to 'insert'
    5 |     ages.insert(10, "John");

Output: {"error_type": "No Matching Member Function", "failure_location": "session_2.cpp:5:17", "code_path": "main() -> std::map::insert(10, \\"John\\")", "confidence": 0.95, "reasoning": "The std::map::insert() function was called with arguments that do not match any valid overload."}

Input: Exception in thread "main" java.lang.NullPointerException: Cannot invoke "String.indexOf(String)" because "recordsList_2" is null
    at MetricsEngine_2.updateCache_2(Api_2.java:110)

Output: {"error_type": "NullPointerException", "failure_location": "Api_2.java:110", "code_path": "main thread -> MetricsEngine_2.updateCache_2() -> String.indexOf()", "confidence": 0.99, "reasoning": "The object 'recordsList_2' is null, so calling the 'indexOf()' method on it causes a NullPointerException."}

Input: CssSyntaxError: grid_2.css:20:1: Unclosed block
  18 |     background: #000;
  19 |     opacity: 0.9;
> 20 |
    | ^
  21 | .element-sibling_2 {

Output: {"error_type": "PostCSS Parser Error: Unclosed Block", "failure_location": "grid_2.css:20:1", "code_path": "CSS parser -> grid_2.css -> unclosed declaration block before '.element-sibling_2'", "confidence": 0.99, "reasoning": "A CSS block was not properly closed with '}', causing the parser to detect an unclosed block before the next selector."}

Input: Error: Stray end tag </span>.
From line 95, column 15; to line 95, column 22
item 1</p></span>\n</di

Output: {"error_type": "W3C Validator Error: Stray End Tag", "failure_location": "Line 95, columns 15-22", "code_path": "HTML parser -> closing tags -> unexpected </span>", "confidence": 0.99, "reasoning": "The closing </span> tag does not match any currently open <span> element, resulting in a stray end tag."}

Input: app_1.ts:63:1 - error TS2322: Type 'string' is not assignable to type 'number'.
63 weightsTensor_1 = "invalid_assign_1";
  ~~~~~~~~~~~~~~~

Output: {"error_type": "TypeScript Error: TS2322 (Type Assignment Error)", "failure_location": "app_1.ts:63:1", "code_path": "Top-level assignment -> weightsTensor_1 = \\"invalid_assign_1\\"", "confidence": 0.99, "reasoning": "A string value was assigned to a variable that is declared with the type 'number', violating TypeScript's type checking rules."}

Input: panic: runtime error: index out of range [7] with length 3
goroutine 1 [running]:
main.main()
    /go/src/app/main_2.go:96 +0x3d

Output: {"error_type": "Go Runtime Panic: Index Out of Range", "failure_location": "/go/src/app/main_2.go:96", "code_path": "main.main() -> slice/array index access", "confidence": 0.99, "reasoning": "The program attempted to access index 7 of a slice or array that contains only 3 elements, causing a runtime panic."}
"""

def triage_agent(title, description, stack_trace="Not available", bug_id="unknown"):
    user_message = f"Title: {title}\nDescription: {description}\nStack Trace: {stack_trace}"
    result = call_llm_with_retry(system_prompt, user_message)
    if result is None:
        result = {"severity": "Medium", "confidence": 0.0, "reasoning": "LLM call failed after retries", "priority": "Medium", "component": "Unknown"}
    result['bug_id'] = bug_id
    return result

def log_analysis_agent(error_text, code_context=""):
    if code_context:
        user_message = f"Code:\n{code_context}\n\nError:\n{error_text}"
    else:
        user_message = f"Error:\n{error_text}"
    result = call_llm_with_retry(log_analysis_system_prompt, user_message)
    if result is None:
        result = {
            "error_type": "Unknown",
            "failure_location": "Not determined",
            "code_path": "Not determined",
            "confidence": 0.0,
            "reasoning": "LLM call failed after retries"
        }
    return result

print("Both agents ready")

Both agents ready


In [12]:
test_title = "App crashes on login"
test_description = "Users report the app crashes immediately after entering valid credentials."
test_stack_trace = """Exception in thread "main" java.lang.NullPointerException: Cannot invoke "String.indexOf(String)" because "recordsList_2" is null
    at MetricsEngine_2.updateCache_2(Api_2.java:110)"""

triage_result = triage_agent(test_title, test_description, test_stack_trace, bug_id="TEST-001")
print("TRIAGE RESULT:")
print(triage_result)

print()

log_result = log_analysis_agent(test_stack_trace)
print("LOG ANALYSIS RESULT:")
print(log_result)

TRIAGE RESULT:
{'bug_id': 'TEST-001', 'severity': 'Critical', 'confidence': 0.9, 'reasoning': 'The app crashes immediately after a critical action (login) with a NullPointerException, indicating a severe issue that completely blocks core functionality.', 'priority': 'Immediate', 'component': 'Login'}

LOG ANALYSIS RESULT:
{'error_type': 'NullPointerException', 'failure_location': 'Api_2.java:110', 'code_path': 'main thread -> MetricsEngine_2.updateCache_2() -> String.indexOf()', 'confidence': 0.99, 'reasoning': "The object 'recordsList_2' is null, so calling the 'indexOf()' method on it causes a NullPointerException."}


In [10]:
# import pandas as pd
# import json
# import time
# import re
# from groq import Groq
# from dotenv import load_dotenv
# import os

# load_dotenv("../.env", override=True)
# api_key = os.getenv("GROQ_API_KEY")
# client = Groq(api_key=api_key)

# print("Setup complete. API key loaded:", api_key[:8] if api_key else "NOT FOUND")import os
# print("Current working directory:", os.getcwd())
# print("Looking for .env at ../.env — exists:", os.path.exists("../.env"))

# 

In [ ]:
# Run this in a new cell - it searches your OWN notebook file for the key prefix
with open("02_triage_agent.ipynb", "r", encoding="utf-8") as f:
    content = f.read()

import re
matches = re.findall(r'gsk_[a-zA-Z0-9]+', content)
print("Number of matches found:", len(matches))
for m in set(matches):
    print(m[:10] + "..." + m[-4:])